### ChromaDB


In [ ]:
# pip install -qU "langchain-chroma>=0.1.2"

In [ ]:
# !pip install langchain chromadb openai tiktoken pypdf langchain_openai langchain-community langchain_google_genai

In [14]:
from dotenv import load_dotenv
load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

In [16]:
from langchain_google_genai.embeddings import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

In [17]:
from langchain_core.documents import Document

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [18]:
# creating a vector store
vector_store = Chroma(
    embedding_function = embedding,
    persist_directory="my-chroma-db",
    collection_name="my-collection"
)

In [44]:
docs = [doc1, doc2, doc3, doc4, doc5]
valid_docs = [doc for doc in docs if doc.page_content and doc.page_content.strip()]

In [ ]:
# add documents in vector store
def batch_add_documents(vector_store, documents, batch_size=1):
    for i in range(0, len(documents), batch_size):
        batch = documents[i : i + batch_size]
        vector_store.add_documents(batch)

batch_add_documents(vector_store, valid_docs)

In [46]:
# view all the dcouments
vector_store.get(include= ["documents"])

{'ids': ['193c514b-4c9e-4ac4-83c0-ab55ed83b9e1',
  '520e5c78-ff2c-494f-9401-3af03944833c',
  '7065f056-a897-44a9-a5cb-8fd3f3949b4b',
  '3c8b375a-456c-4c5a-a94f-28ee7ac864e5',
  '097ecec1-6b94-4019-a04b-cc900be503f0'],
 'embeddings': None,
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
  'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.',
  'Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.',
  'Ravindra Jadeja is a dynamic al

In [37]:
# search documents
vector_store.similarity_search(
    query = "Who amomg of these are bowlers",
    k = 1 # it gives no of outputs that have the similarity with the query
)

[Document(id='17c591d9-43e8-4b7f-95df-576739b4dcc0', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

In [47]:
# search with similarity score
vector_store.similarity_search_with_score(
    query = "Who among of these are bowler?" # score is the distance between two vectors.
)

[(Document(id='3c8b375a-456c-4c5a-a94f-28ee7ac864e5', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.6092575192451477),
 (Document(id='097ecec1-6b94-4019-a04b-cc900be503f0', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.6640057563781738),
 (Document(id='193c514b-4c9e-4ac4-83c0-ab55ed83b9e1', metadata={'team': 'Royal Challengers Bangalore'}, page_content='Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.'),
  0.7100850343704224),
 (Document(id='520e5c78-ff2c-494f-9401-3af039448

In [38]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query = "M",
    filter = {"team": "Mumbai Indians"}
)

[(Document(id='17c591d9-43e8-4b7f-95df-576739b4dcc0', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.8455047011375427),
 (Document(id='da61d340-0c70-4bb7-ab01-03ef66e272f4', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.8455047011375427),
 (Document(id='77d50a83-934e-4a58-96fc-317a23528660', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  0.8611332178115845),
 (Document(id='f8f89f17-5350-4ecf-8499-eacbe5023a8a', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Shar

In [30]:
# update documents
updated_doc1 = Document(
    page_content = "Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata = {"team": "Royal Challengers Bangalore"}
    )

vector_store.update_document(document_id= "099150c0-bb44-45bd-9a3a-d926f797d8d2", document= updated_doc1)

In [ ]:
# delete documents
vector_store.delete(ids=["099150c0-bb44-45bd-9a3a-d926f797d8d2"])

# delete all ids
'''all_ids = vector_store.get()['ids']

# Delete if there are any IDs present
if all_ids:
    vector_store.delete(ids=all_ids)'''

In [48]:
vector_store.get(include = ["documents", "metadatas", "embeddings"])

{'ids': ['193c514b-4c9e-4ac4-83c0-ab55ed83b9e1',
  '520e5c78-ff2c-494f-9401-3af03944833c',
  '7065f056-a897-44a9-a5cb-8fd3f3949b4b',
  '3c8b375a-456c-4c5a-a94f-28ee7ac864e5',
  '097ecec1-6b94-4019-a04b-cc900be503f0'],
 'embeddings': array([[ 0.0074764 ,  0.01034819,  0.01265225, ...,  0.00313006,
          0.01334844,  0.00201392],
        [ 0.00578349,  0.03275334, -0.00712996, ...,  0.00599   ,
          0.00842474,  0.01284763],
        [ 0.00278233,  0.02150997, -0.00733589, ..., -0.00771123,
          0.01547233,  0.01310577],
        [ 0.03408059,  0.01119854, -0.01184934, ...,  0.00848975,
          0.02335239, -0.0077354 ],
        [-0.00296271,  0.01466555, -0.01345706, ...,  0.00715038,
          0.01247527,  0.00910379]], shape=(5, 3072)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the m